# KrishiMitra — Crop Sowing Recommender (Model 2)

OCI Data Science notebook for the **Smart Sowing Advisor** (Module 2).

- **Algorithm:** Gradient Boosting (XGBoost)
- **Features (14):** district avg rainfall (5y), soil type (encoded), sowing month, current Mandi price trend (30d), historical yield (3y district avg), land size, etc.
- **Targets:** recommended crop category + expected yield (qtl/acre)
- **Evaluation:** RMSE, R², 5-fold cross-validation
- **Explainability:** SHAP
- **Deployment:** OCI Data Science Model Deployment REST endpoint (see `score.py`)

> Training data in production combines the ICRISAT district dataset, Agmarknet historical prices, and IMD historical rainfall. Here we synthesise a representative dataset so the notebook runs end-to-end; swap `load_training_data()` for the real joined extract from Oracle ATP / Object Storage.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import xgboost as xgb

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load training data
Replace with the real ATP/Object Storage extract in production.

In [ ]:
SOIL_TYPES = ['Sandy', 'Loamy', 'Clay', 'Black', 'Silt']
CROPS = ['Wheat', 'Rice', 'Maize', 'Cotton', 'Soybean', 'Mustard', 'Gram']
DISTRICTS = ['Lucknow', 'Kanpur', 'Bhopal', 'Indore', 'Ludhiana', 'Amritsar']

def load_training_data(n=6000):
    df = pd.DataFrame({
        'district': np.random.choice(DISTRICTS, n),
        'soil_type': np.random.choice(SOIL_TYPES, n),
        'sow_month': np.random.randint(1, 13, n),
        'rainfall_5y_mm': np.random.normal(900, 250, n).clip(200, 2000),
        'price_trend_30d_pct': np.random.normal(0, 8, n),
        'hist_yield_3y': np.random.normal(18, 5, n).clip(5, 40),
        'land_acres': np.random.gamma(2.0, 2.0, n).clip(0.5, 25),
        'temp_avg_c': np.random.normal(27, 5, n).clip(8, 42),
        'humidity_pct': np.random.normal(60, 15, n).clip(10, 95),
        'irrigation_idx': np.random.uniform(0, 1, n),
    })
    df['crop'] = np.random.choice(CROPS, n)
    # Synthetic yield as a function of the features (+ noise) so the model has signal.
    base = (df['hist_yield_3y'] * 0.6
            + df['rainfall_5y_mm'] * 0.004
            + df['irrigation_idx'] * 6
            - (df['temp_avg_c'] - 27).abs() * 0.3
            + df['price_trend_30d_pct'] * 0.05)
    df['expected_yield_qtl_acre'] = (base + np.random.normal(0, 1.5, n)).clip(3, 45).round(2)
    return df

df = load_training_data()
df.head()

## 2. EDA

In [ ]:
print(df.describe(numeric_only=True).T)
print('\nNulls:\n', df.isna().sum())
print('\nYield by crop:\n', df.groupby('crop')['expected_yield_qtl_acre'].mean().sort_values(ascending=False))

## 3. Feature engineering + train/test split

In [ ]:
CAT_FEATURES = ['district', 'soil_type', 'crop']
NUM_FEATURES = ['sow_month', 'rainfall_5y_mm', 'price_trend_30d_pct', 'hist_yield_3y',
                'land_acres', 'temp_avg_c', 'humidity_pct', 'irrigation_idx']
TARGET = 'expected_yield_qtl_acre'

X = df[CAT_FEATURES + NUM_FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

preprocess = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_FEATURES),
    ('num', 'passthrough', NUM_FEATURES),
])

## 4. Train XGBoost regressor

In [ ]:
model = Pipeline([
    ('prep', preprocess),
    ('xgb', xgb.XGBRegressor(
        n_estimators=400, max_depth=6, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, random_state=RANDOM_STATE, n_jobs=-1)),
])
model.fit(X_train, y_train)

## 5. Evaluation (RMSE, R², 5-fold CV)

In [ ]:
pred = model.predict(X_test)
rmse = mean_squared_error(y_test, pred, squared=False)
r2 = r2_score(y_test, pred)
print(f'Test RMSE: {rmse:.3f} qtl/acre')
print(f'Test R2:   {r2:.3f}')

cv = cross_val_score(model, X, y, cv=KFold(5, shuffle=True, random_state=RANDOM_STATE),
                     scoring='r2')
print(f'5-fold CV R2: {cv.mean():.3f} +/- {cv.std():.3f}')

## 6. SHAP explainability

In [ ]:
import shap
X_test_enc = model.named_steps['prep'].transform(X_test)
feat_names = CAT_FEATURES + NUM_FEATURES
explainer = shap.TreeExplainer(model.named_steps['xgb'])
shap_values = explainer.shap_values(X_test_enc)
shap.summary_plot(shap_values, X_test_enc, feature_names=feat_names, show=True)

## 7. Save model to the OCI Model Catalog

In [ ]:
import joblib, os
os.makedirs('model-artifact', exist_ok=True)
joblib.dump(model, 'model-artifact/sowing_model.joblib')
print('Saved model-artifact/sowing_model.joblib')

# In OCI Data Science, register + deploy with ADS:
#   from ads.model.framework.sklearn_model import SklearnModel
#   m = SklearnModel(estimator=model, artifact_dir='model-artifact')
#   m.prepare(inference_conda_env='generalml_p38_cpu_v1', score_py='score.py')
#   m.verify(X_test.head().to_dict(orient='records'))
#   model_id = m.save(display_name='krishimitra-sowing-recommender')
#   m.deploy(display_name='krishimitra-sowing-md')